In [17]:
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
from scipy.interpolate import griddata
import pyproj 
import os

Cargamos los datos municipales

In [18]:
base_dir = os.getcwd()+ r"\DATA"
ruta_shapefile = base_dir + r"\MUNICIPIOS\MUNICIPIOS\00mun.shp"
gdf_munis = gpd.read_file(ruta_shapefile)

Cargamos el DF de ingresos X municipio 

#NOTA:
___ 

Si deseamos hacer un raster con otra columna del df, basta con sustituir la variable:
     columna_valor

In [19]:
# ---------------------------------------------------------
# 1. PREPARAR TU DATAFRAME
# ---------------------------------------------------------
ruta_IngresosXMunicipios = base_dir +r"\DatosEconomicos\INGRESOS_MUNICIPIO.csv"
df = pd.read_csv(ruta_IngresosXMunicipios )
df["codigo"] = df['estado'] * 1000 + df['municipio']

columna_valor = 'EstimPagoPromedio' 

In [20]:
base_dir

'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA'

In [21]:
gdf_munis = gpd.read_file(base_dir+r"\MUNICIPIOS\MUNICIPIOS\00mun.shp")
gdf_munis['estado_shp'] = gdf_munis['CVE_ENT'].astype(int)
gdf_munis['mun_shp'] = gdf_munis['CVE_MUN'].astype(int)
gdf_munis['codigo'] = gdf_munis['estado_shp'] * 1000 + gdf_munis['mun_shp']


gdf_merged = gdf_munis.merge(df, on='codigo', how='inner')
print("Reproyectando a Lambert Conformal Conic...")
# Cadenas de proyección

proj_lcc_str = "+proj=lcc +lat_1=17.5 +lat_2=29.5 +lat_0=12 +lon_0=-102 +x_0=2500000 +y_0=0 +ellps=GRS80 +units=m +no_defs"
gdf_merged = gdf_merged.to_crs(proj_lcc_str)

# ---------------------------------------------------------
# 5. CONFIGURAR LA RESOLUCIÓN Y DIMENSIONES DEL RASTER
# ---------------------------------------------------------
print("Configurando la malla de 1km...")
resolucion = 1000 # 1000 metros = 1 km por píxel

# Obtener los límites extremos de todo México en esta proyección
xmin, ymin, xmax, ymax = gdf_merged.total_bounds

# Calcular cuántos píxeles de ancho y alto tendrá la imagen
width = int(np.ceil((xmax - xmin) / resolucion))
height = int(np.ceil((ymax - ymin) / resolucion))

# Crear la transformación afín (indica dónde empieza el raster y de qué tamaño es el píxel)
transform = from_origin(xmin, ymax, resolucion, resolucion)

# ---------------------------------------------------------
# 6. RASTERIZAR LOS POLÍGONOS
# ---------------------------------------------------------
print("Generando el Raster (esto puede tomar unos segundos)...")

# Crear un generador de pares (Geometría, Valor_a_quemar)
# Llenamos los valores nulos con 0 para evitar errores
gdf_merged[columna_valor] = gdf_merged[columna_valor].fillna(0)
shapes = ((geom, value) for geom, value in zip(gdf_merged.geometry, gdf_merged[columna_valor]))

# Ejecutar la rasterización
imagen_raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,               # Valor de fondo (océano/fuera de México)
    all_touched=False,    # Al ser False, asigna el valor si el centro del pixel cae en el municipio
    dtype=rasterio.float32 # Usamos float32 por si tus datos tienen decimales
)

# ---------------------------------------------------------
# 7. EXPORTAR A QGIS (GEOTIFF)
# ---------------------------------------------------------
nombre_salida = os.getcwd()+r'\Resultados\raster_municipios.tif'

with rasterio.open(
    nombre_salida,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype=imagen_raster.dtype,
    crs=proj_lcc_str,
    transform=transform,
    nodata=0
) as dst:
    dst.write(imagen_raster, 1)

print(f"¡Listo! Archivo {nombre_salida} generado exitosamente a 1km de resolución.")

Reproyectando a Lambert Conformal Conic...
Configurando la malla de 1km...
Generando el Raster (esto puede tomar unos segundos)...
¡Listo! Archivo c:\Users\xboxn\Documents\QgisProy\Resultados\raster_municipios.tif generado exitosamente a 1km de resolución.


Ahora pasaremos aprocesar el segundo conjunto de datos que tiene el porcentaje de pobreza en algunas localidades

In [23]:
# Definimos la proyección Lambert
proj_lcc_str = "+proj=lcc +lat_1=17.5 +lat_2=29.5 +lat_0=12 +lon_0=-102 +x_0=2500000 +y_0=0 +ellps=GRS80 +units=m +no_defs"

gdf_munis = gpd.read_file(ruta_shapefile)


gdf_munis = gdf_munis.to_crs(proj_lcc_str)
xmin, ymin, xmax, ymax = gdf_munis.total_bounds


df = pd.read_csv(base_dir + r'\DatosEconomicos\Pobreza.csv')
columna_valor = 'pobreza'

# Limpiamos filas con nulos
df = df.dropna(subset=['LON_DECIMAL', 'LAT_DECIMAL', columna_valor])

# Transformador de WGS84 (grados) a LCC (metros)
transformer = pyproj.Transformer.from_crs("EPSG:4326", proj_lcc_str, always_xy=True)

# Convertimos las coordenadas de los puntos a metros (LCC)
puntos_lon = df['LON_DECIMAL'].values
puntos_lat = df['LAT_DECIMAL'].values
puntos_x_lcc, puntos_y_lcc = transformer.transform(puntos_lon, puntos_lat)

# Unimos X e Y en el formato que espera SciPy
puntos = np.column_stack((puntos_x_lcc, puntos_y_lcc))
valores = df[columna_valor].values

# ---------------------------------------------------------
# 3. CONFIGURAR LA MALLA
# ---------------------------------------------------------
print("Generando la malla de 1km en proyección Lambert...")
resolucion_metros = 1000 

width = int(np.ceil((xmax - xmin) / resolucion_metros))
height = int(np.ceil((ymax - ymin) / resolucion_metros))

# Ajustar los límites máximos basados en la cantidad exacta de píxeles
xmax_ajustado = xmin + width * resolucion_metros
ymin_ajustada = ymax - height * resolucion_metros

# Crear las coordenadas matemáticas para los centros de cada píxel en metros
grid_x_1d = np.linspace(xmin + resolucion_metros/2, xmax_ajustado - resolucion_metros/2, width)
grid_y_1d = np.linspace(ymax - resolucion_metros/2, ymin_ajustada + resolucion_metros/2, height)

grid_x, grid_y = np.meshgrid(grid_x_1d, grid_y_1d)

# ---------------------------------------------------------
# 4. INTERPOLACIÓN: ASIGNAR EL PUNTO MÁS CERCANO A CADA PÍXEL
# ---------------------------------------------------------
print("Calculando el vecino más cercano para cada píxel (esto tomará algo de tiempo)...")

# Calculamos el vecino más cercano ahora usando metros de distancia
Z_raster = griddata(puntos, valores, (grid_x, grid_y), method='nearest')
Z_raster = Z_raster.astype(np.float32)

# ---------------------------------------------------------
# 5. EXPORTAR A QGIS 
# ---------------------------------------------------------
print("Exportando a archivo GeoTIFF...")
nombre_salida = os.getcwd()+r'\Resultados\raster_pobreza.tif'

# Establecer que la imagen empieza en la esquina superior izquierda
transform = from_origin(xmin, ymax, resolucion_metros, resolucion_metros)

with rasterio.open(
    nombre_salida,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype=Z_raster.dtype,
    crs=proj_lcc_str, # <--- Asignamos la proyección Lambert al archivo de salida
    transform=transform,
    nodata=-9999  # Valor nulo
) as dst:
    dst.write(Z_raster, 1)

print(f"¡Listo! Archivo '{nombre_salida}' generado exitosamente en Lambert Conformal Conic. Puedes arrastrarlo a QGIS.")

Generando la malla de 1km en proyección Lambert...
Calculando el vecino más cercano para cada píxel (esto tomará algo de tiempo)...
Exportando a archivo GeoTIFF...
¡Listo! Archivo 'c:\Users\xboxn\Documents\QgisProy\Resultados\raster_pobreza.tif' generado exitosamente en Lambert Conformal Conic. Puedes arrastrarlo a QGIS.
